In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import numpy as np
import matplotlib.pyplot as plt
from xml.etree import ElementTree
import os
import cv2
import random

In [ ]:
class_names = ["trafficlight", "stop", "crosswalk", "speedlimit"]
class_names_label = {class_name: i for i, class_name in enumerate(class_names)}

n_classes = 4
size = (300,400)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def load_data():
    gdrive = 'drive/My Drive/archive_2/'
    datasets = ['train', 'test', 'val']
    output = []

    for dataset in datasets:
        imags = []
        labels = []
        directoryA = gdrive + dataset +"/annotations/"
        directoryIMG = gdrive + dataset +"/images/"
        file = os.listdir(directoryA)
        img = os.listdir(directoryIMG)
        file.sort()
        img.sort()

        i = 0
        for xml in file:

            xmlf = os.path.join(directoryA,xml)
            dom = ElementTree.parse(xmlf)
            vb = dom.findall('object')
            label = vb[0].find('name').text
            labels.append(class_names_label[label])

            img_path = directoryIMG + img[i]
            curr_img = cv2.imread(img_path)
            curr_img = cv2.resize(curr_img, size)
            imags.append(curr_img)
            i +=1

        imags = np.array(imags, dtype='float32')
        imags = imags / 255

      #  labels = pd.DataFrame(labels)
        labels = np.array(labels, dtype='int32')

        output.append((imags, labels))
    return output


In [ ]:
(train_images, train_labels), (test_images, test_labels), (val_images, val_labels) = load_data()

In [ ]:
print(len(train_images))

613


In [ ]:
class TSignDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.conv3 = nn.Conv2d(64, 128, 3)
        self.conv4 = nn.Conv2d(128, 256, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(94208, 128)
        self.fc2 = nn.Linear(128, 4)

        self.loss_fn = nn.NLLLoss()

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.dropout(x)
        x = self.pool(F.relu(self.conv4(x)))
        x = self.dropout(x)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        output = F.log_softmax(x, dim=0)
        return output

In [ ]:
from torchvision.transforms import v2

In [ ]:
# Data Augmentation
img_transforms = v2.Compose([
    v2.ColorJitter(),
    v2.RandomChannelPermutation(),
    v2.RandomGrayscale(),
])
# img_transforms = v2.Compose([
#     v2.RandomResizedCrop(size=size),
#     v2.RandomHorizontalFlip(p=0.5),
#     v2.ToDtype(torch.float32, scale=True),
#     v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

class CustomDataset(Dataset):
  def __init__(self, images, labels, transforms=None):
    self.images = images
    self.labels = labels
    self.transform = transforms

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image = self.images[idx].permute(2, 0, 1)
    label = self.labels[idx]
    if self.transform:
      image = self.transform(image)
    return image, label

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_images = torch.tensor(train_images, device='cuda')
train_labels = torch.tensor(train_labels, dtype=torch.long, device='cuda')
train_dataset = CustomDataset(train_images, train_labels, transforms=img_transforms)

# generator = torch.Generator(device='cuda')
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True) # , generator=generator

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
val_images = torch.tensor(val_images).to(device)
val_labels = torch.tensor(val_labels, dtype=torch.long).to(device)
val_dataset = CustomDataset(val_images, val_labels)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True) # , generator=generator

test_images = torch.tensor(test_images).to(device)
test_labels = torch.tensor(test_labels, dtype=torch.long).to(device)
test_dataset = CustomDataset(test_images, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True) # , generator=generator

In [ ]:
def get_acc(model, loader):
    total, correct = 0, 0
    model.eval()
    for inputs, labels in loader:
        # inputs = inputs.permute(0, 3, 1, 2)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return correct/total

In [ ]:
def train():
    model = TSignDetector().to(device)
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 50
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            inputs = inputs.to(device)
            outputs = model(inputs)
            loss = model.loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        if epoch % 5 == 0:
          print(f"Epoch {epoch}, Loss: {running_loss}")
          print("Train Acc: ", get_acc(model, train_loader))
          print("Val Acc: ", get_acc(model, val_loader))
          print("\n")
    return model

model = train()

Epoch 0, Loss: 106.28377938270569
Train Acc:  0.41435562805872755
Val Acc:  0.42748091603053434


Epoch 5, Loss: 101.24666452407837
Train Acc:  0.6247960848287113
Val Acc:  0.5190839694656488


Epoch 10, Loss: 92.55081880092621
Train Acc:  0.7438825448613376
Val Acc:  0.6106870229007634


Epoch 15, Loss: 85.5474066734314
Train Acc:  0.8287112561174551
Val Acc:  0.6412213740458015


Epoch 20, Loss: 82.92004108428955
Train Acc:  0.867862969004894
Val Acc:  0.6412213740458015


Epoch 25, Loss: 79.79036697745323
Train Acc:  0.8727569331158238
Val Acc:  0.6259541984732825


Epoch 30, Loss: 79.54687869548798
Train Acc:  0.8809135399673735
Val Acc:  0.6335877862595419


Epoch 35, Loss: 78.93054270744324
Train Acc:  0.9053833605220228
Val Acc:  0.6641221374045801


Epoch 40, Loss: 78.90571689605713
Train Acc:  0.8711256117455138
Val Acc:  0.7022900763358778


Epoch 45, Loss: 79.98572552204132
Train Acc:  0.9184339314845025
Val Acc:  0.6030534351145038




In [ ]:
print("Test")
runs = 10
acc = 0
for i in range(runs):
  acc += get_acc(model, test_loader)
print(acc/runs)

Test
0.7127819548872181


In [ ]:
# Save model to Drive
model_name = 'classifier.pt'
path = f"/content/drive/My Drive/archive_2/{model_name}"
torch.save(model.state_dict(), path)